<img src=images/gdd-logo.png width=300px align=right>

# Data Processing

In this notebook you are going to see how you can apply different kinds of preprocessing steps to your data. In particular we will look at:

1. [Data Preparation](#prep)
2. [Variable Encoding](#enc)
3. [Feature Scaling](#feat)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

## The Data

In this notebook we will work with the [Titanic dataset](https://www.kaggle.com/c/titanic/data), containing historical data about Titanic passengers and whether they survived the wreck.

<img src="images/Data_Processing/titanic.png" style="display: block;margin-left: auto;margin-right: auto;width: 600px"/>

We will first focus on sklearn's preprocessing features, but before we can go ahead, let's load the dataset and get it ready for it.

In [ ]:
titanic_df = pd.read_csv('data/titanic.csv')
titanic_df.head()

## <mark> Exercise: Data Exploration </mark>

Take some time to examine the dataset and answer the following questions:

1. How many data points do you have and how many features? 

In [ ]:
# your code.

2. What do the features represent (i.e. what is the meaning of each column)? 

You can find more information about some of the columns [here](https://www.kaggle.com/c/titanic/data). 

In [ ]:
# your code.

3. What data types do you have per feature? Integers, floats, booleans, strings? Will this influence model building? 

In [ ]:
# your code.

4. Are there any missing values? Would this influence model building? 

In [ ]:
# your code.

5. Which features do you think are relevant? Is there any redundant information? How can you be sure?

In [ ]:
# your code.

   
**<font color='green'> Bonus Challenge: Pandas </font>**

Produce some summary statistics for the different features. Are any of the features correlated? Group the data by the `survived` column and compare statistics.

In [ ]:
# your code.

**<font color='green'> Bonus Challenge: Machine Learning </font>** 

What machine learning algorithms would you consider for this problem? Why have you made this choice?

In [ ]:
# your code.

<a id='prep'></a>
## Data Preparation 

Raw datasets are often not suitable for machine learning algorithms. For example, the dataset -- like this Titanic dataset! --  may contain categorical features or have missing values. Preprocessing the dataset is therefore an import phase of a project to ensure that machine learning is feasible.

However, let's first remove the redundant columns, create our feature matrix **X** and target vector **y**, and split the data intro training and test sets.

In [ ]:
def drop_unwanted_cols(df, cols):
    return df.drop(columns=cols)

def create_Xy(df, target='survived'):
    df = df.reset_index(drop=True)
    return df.drop(columns=[target]), df[target]

We used the `.pipe` functionality in Pandas to apply our written preprocessing functions to our titanic dataframe. This structures our code more neatly. See the [documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pipe.html) for more information on `.pipe`. 

In [ ]:
unwanted_cols = ['embarked', 'sex', 'adult_male',
                 'deck', 'alive', 'class']

X, y = (
    titanic_df
    .pipe(drop_unwanted_cols, cols=unwanted_cols)
    .dropna()
    .pipe(create_Xy)
)

Let's check if the data is what you would expect.

In [ ]:
print(X.shape, y.shape)

In [ ]:
X.head()

In [ ]:
y.head()

🔔 Lastly, **before** you start further processing the data, it is important to split it into a training and a testing sets to avoid **data leaks**.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=111)

print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

You want to avoid **any** information from the test set being used during training, and that includes preprocessing steps.
This is to ensure that the testing set is a good representation of how it would be to use your model with new data.

If information from the test set *does* leak into the training data (e.g. you scale the data using information from the complete dataset), it may cause your metrics to overestimate the model's performance.

<a id='enc'></a>
## Variable Encoding 


Unfortunately, this dataset needs a little more preprocessing before you can use it to fit a model. Let's demonstrate: 

In [ ]:
# from sklearn.svm import SVC

# model = SVC()
# model.fit(X_train, y_train)

The error you get is ```ValueError: could not convert string to float: 'man'```. This is because the dataset contains **categorical** variables that are encoded as strings, rather than numbers. For example, the column `who` contains the strings "woman", "man" and "child".

It is possible to encode these categorical variables into numeric ones. Fortuantely, Scikit-Learn makes it easy for you to do so.

<a id='ordinal'></a>
### Ordinal Encoding
🔔 The `OrdinalEncoder` can numerically encode columns. Much like the predictive models you worked with previously, it must first be imported. 

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

Then it must be instantiated. This bears some similarity to how you would instantiate a machine learning model with `model = DecisionTreeClassifier()`.

In [ ]:
ordinal_encoder = OrdinalEncoder()

In Scikit-Learn, much like the Machine Learning Models you have already seen, the preprocessing algorithms are implemented as Python objects. 

They are referred to as **Transformers**: 

- Transformers have a `.fit()` method implemented (just like the Models we have already seen). This allows the transformer to learn the _parameters_ necessary to transform data. 
- Transformers also have a `.transform()` method. This method is used to perform the transformation (after the _parameters_ have been learnt through te `.fit()` method. 

You can think of Transformers as anything that _transforms_ your data. Let's demonstrate this on some categorical columns of the dataset:

In [ ]:
categorical_columns = ['who', 'embark_town']
X_train[categorical_columns]

In [ ]:
# Fit the encoder. 
ordinal_encoder.fit(X_train[categorical_columns])

In [ ]:
# Transform the data.
encoded_column = ordinal_encoder.transform(X_train[categorical_columns])

In [ ]:
X_train[categorical_columns].head()

In [ ]:
encoded_column[0:5]

_So what happened?_


#### Where are the column headings?

The output is a NumPy array; the data type used to store the data has changed. This change occurs by default because it is more efficient for scikit-learn to work with NumPy arrays (since Pandas is built on top of NumPy).

Running the code below will make scikit-learn return Pandas DataFrames - meaning the output will be easier for us humans to understand!

In [ ]:
from sklearn import set_config
set_config(transform_output = "pandas")

In [ ]:
# Fit the encoder. 
ordinal_encoder.fit(X_train[categorical_columns])

# Transform the data.
encoded_column = ordinal_encoder.transform(X_train[categorical_columns])

# View the top 5 rows
encoded_column.head()

#### **Fitting the Transformer** 
During the `fit` step the Transformer creates a mapping from the categorical values (strings) in the data to a numeric value. For example: in the `embark_town` column, there are three values: _Cherbourg_, _Queenstown_ (_Cobh_) and _Southampton_. When we call `.fit`, a mapping is created from these original strings to their numeric respresentations: 0, 1 and 2 respectively. 

However, this did not change any data yet! It simply learned the associations between the string (e.g. _Cherbourg_) and the corresponding integer (e.g. 0). 

| Original | Ordinal  |
|----------------|-----------|
| Cherbourg      | 0         |
| Queenstown     | 1         |
| Southampton    | 2         |

#### **Transforming the data** 
During the `transform` step, the created mapping is applied to the data. This will return the data with a numeric representation. *Notice how our output is now no longer a Pandas dataframe, but a numpy array?* 

So why are fit and transform separate steps? Well, the learned mapping can be applied to the original data -- but also to new data! 

To demonstrate, let's create some new arbitrary data points that also have a `who` and `embark_town` column. 

In [ ]:
data = {
    'who': ['woman', 'man', 'child', 'child'],
    'embark_town': ['Queenstown', 'Southampton', 'Cherbourg', 'Queenstown']
}

new_datapoints = pd.DataFrame(data)
new_datapoints

In [ ]:
ordinal_encoder.transform(new_datapoints)

As you can see, the mapping can be applied to new data as well!

**Tips & tricks** 
`.fit_transform()` is an implemented method that applies the data transformation directly after the mapping has been created. The output is equivalent to first performing `fit` and then `transform` directly after. 

In [ ]:
encoded_columns = ordinal_encoder.fit_transform(X_train[categorical_columns])
encoded_columns[0:5]

As the mapping is created during `fit_transform`, it can still be used on new data as well.

In [ ]:
ordinal_encoder.transform(new_datapoints)

<a id='onehot'></a>
### One-hot encoding features

**Question:** What issues could you get from using an ordinal encoder on some of the categorical values?

<details>
    
  <summary><span style="color:blue">Show answer</span></summary>
  
This way of encoding categorial values may work for problems where there is a natural ordinal relationship between the categories, such as medals in the Olympics. Gold and Silver are more similar than Gold and Bronze, and this (ordinal) relationship is preserved if Gold, Silver and Bronze are mapped to 0, 1 and 2 respectively. 

However, imagine your categories are <font color='red'>red</font>, <font color='blue'>blue</font> and <font color='yellow'>yellow</font>. If red, blue and yellow are mapped to 0, 1, and 2 respectively, we are introducing a new relationship between these values (e.g. yellow and red are more different than yellow and blue) that does not exist. 

Label encoding categorical values when there is no ordinal relationship can case cause low performance or unexpected results. 

An alternative approach is to use one-hot or dummy encoding. This type of encoding can be obtained with Scikit-Learn's `OneHotEncoder`.

</details>



## <mark> Exercise: One Hot Encoder </mark>

Use Scikit-Learn's `OneHotEncoder` to one-hot encode the data. 

The one-hot encoder, like the ordinal encoder, is also a **Transformer** in Scikit-Learn. This means it also has a `fit` method to learn a mapping and a `transform` method to transform the data. 

Steps:
* Import the One-hot Encoder (already done for you) 
* Instantiate the One-hot Encoder by setting `sparse_output=False` (or `sparse=False` for sklearn version <1.4)
* Use the `.fit` method to create the mapping
* Use the `.transform` method to transform the categorical data


Check out the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) for more information. 

**<font color='green'> Bonus Challenge: What happened? </font>** 

How many output columns do you now have? Can you explain what happened? How is one-hot encoding different from ordinal encoding? 

**<font color='green'> Bonus Challenge: drop first. </font>** 

Use the `drop='first'` option when initialising the OneHotEncoder. What changes? 

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# add your code

In [ ]:
# %load answers/06_Data_Processing/onehot.py

🔔  The OneHotEncoder transforms categorical values into vectors containing zeroes and ones. The length of the vectors is equal to the number of categories that the model needs to classify. Each element in the vector corresponds to one of the categories, which is 1. All other elements are 0. 

| Chebourg  | Queenstown | Southampton|
|-----------|------------|------------|
| 1         | 0          | 0          |
| 0         | 1          | 0          |
| 0         | 0          | 1          |

An individual feature column (e.g. _embark_town_) is now represented by _n_ (e.g. three for _embark_town_) separate new columns with binary (0 or 1) indicators. However, this is a little excessive. For example, because there are only three possible embarking locations, if we know that someone _didn't_ embarked from Queenstown **and** _didn't_ embark from Southampton, they must have embarked from Cherbourg.

Therefore, we use the `drop='first'` option in OneHotEncoder, which ensures that we keep *n-1* dummies per categorical variable.

| Queenstown | Southampton|
|------------|------------|
| 0          | 0          |
| 1          | 0          |
| 0          | 1          |

<a id='coltrans'></a>
### Column Transformer

With both the ordinal encoder and the one-hot encoder, you only transformed the categorical columns. What happens if you transform _all_ the columns? 

In [ ]:
# Initialise encoder.
onehot_encoder = OneHotEncoder(sparse_output=False, drop='first')

# Fit the encoder. 
onehot_encoder.fit(X_train)

# Transform the data.
encoded_columns = onehot_encoder.transform(X_train)
encoded_columns.shape

It seems like all columns are treated as categorical; however, you do not want columns like _age_ or _fare_ to be categorical! The original values need to be preserved. 

This is where the `ColumnTransformer` comes in. This allows you to apply a certain Transformer, like the one hot encoder or ordinal encoder, on only the specified columns rather than all columns. Then, you can specifically state what we want to do with the remaining columns: _drop_ or _passthrough_. With passthrough, the original values are preserved. 

In [ ]:
from sklearn.compose import ColumnTransformer 

column_transformer = ColumnTransformer([
    ('ordinal_encoder', OrdinalEncoder(), categorical_columns),
    ], remainder="passthrough")

X_encoded = column_transformer.fit_transform(X_train)
X_encoded.shape

In [ ]:
X_encoded

Note that the ColumnTransformer will affect the order of your columns!

## <mark> Exercise: ColumnTransformer </mark>
Use the column transformer for one hot encoding and look at the shape of the data.

In [ ]:
# add your code here

In [ ]:
# %load answers/06_Data_Processing/column_transformer.py

### Putting it all together

Let's build a baseline model with a one-hot encoder and a Support Vector Classifier. 

Notice that the code calls `.fit_transform()` on the training data, but `.transform()` on the testing data!

In [ ]:
from sklearn.compose import ColumnTransformer 
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC

# Create train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=111)

# List categorical columns
categorical_columns = ['who', 'embark_town']

# One-hot encode the data
column_transformer = ColumnTransformer([
    ('one_hot_encoder', OneHotEncoder(sparse_output=False, drop='first'), categorical_columns)
    ], remainder="passthrough")

X_train_encoded = column_transformer.fit_transform(X_train)
X_test_encoded = column_transformer.transform(X_test)

# Model
model = SVC()
model.fit(X_train_encoded, y_train)

# Evaluate
predictions = model.predict(X_test_encoded)
accuracy_score(predictions, y_test)

Quite nice! You were able to fit a model and create predictions. However, you are building a Support Vector Machine, which is heavily reliant on the distance between the values of the feature matrix. 

Therefore you can use feature scaling to ensure that the sensitivity of the scale of your features makes sense.

<a id='feat'></a>
## Feature Scaling

🔔  Distance-based algorithms like support vector machines or k-nearest neighbors are very sensitive to the **scale** of the variables. 

Imagine a dataset that has an _age_ variable and an _income_ variable. The range for _age_ in this dataset is 25-65, while the _income_ in this dataset can vary between \\$25.000 and \\$150.000. When the algorithm calculates the distance between two datapoints, the values of _income_ have a greater impact on the total distance than the values of _age_. Conversely, a change of 25 years in _age_ has the same impact on the total distance as a change of income of _25$_. This does not seem right.

Many machine learning algorithms therefore perform better when numerical input variables are transformed into a standardized scale.

### Feature scaling with Scikit-Learn

Scikit-Learn has a couple of in-built feature scalers available: 
* [**Standard scaler**](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html): standardizes each feature value by removing the _mean_ and scaling by the _variance_ of that particular feature. 
* [**Robust scaler**](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html#sklearn.preprocessing.RobustScaler): standardizes each feature by removing the median and scaling the data according to a specified quantile range. This scaler is a little more robust to outliers, but at the expense of changing the distribution of the data. Note that by default, the quantile range is set (25, 75), which is known as the interquartile range (IQR). You can find more information [here](https://en.wikipedia.org/wiki/Quantile).
* [**Min-max scaler**](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html): transforms all features to a given range by subtracting the minimum value of the feature and dividing by the range. The default range is 0-1. 

Let's try it out.

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

In [ ]:
scaler.fit(X_train_encoded) 

X_train_scaled = scaler.transform(X_train_encoded)

The standard scaler will ensure that all features have a zero mean and standard deviation of 1. Let's see if this is the case.

In [ ]:
# Zero mean
X_train_scaled.mean(axis=0).round(2)

In [ ]:
# Standard deviation of 1
X_train_scaled.std(axis=0).round(2)

By using the `.transform()` on the test set,  you ensure it is encoded in the same way as the train set.

In [ ]:
X_test_scaled = scaler.transform(X_test_encoded)

In [ ]:
X_test_scaled.mean(axis=0).round(2)

In [ ]:
X_test_scaled.std(axis=0).round(2)

## <mark> Exercise: scale and evaluate the model </mark>
Let's see how well the model is doing with scaled data!

1. Try either the `MinMaxScaler` or `RobustScaler` to scale the data. 
2. Train the model using the scaled data.
3. Make predictions and evaluatie how well the model is doing using the accuracy_score function. 

In [ ]:
# add your code here


**<font color='green'> Bonus Challenge: Which scaler to choose? </font>** 

Can you think of reasons or scenarios where one scaler might be a better choice than another? What scaler would be most appropriate for this particular dataset? 

<details>
    
  <summary><span style="color:blue">Show answer</span></summary>
  
- The StandardScaler is a good default choice. This scaler is however relatively sensitive to the presence of outliers.

- The MinMaxScaler is also sensitive to outliers. This one is often picked if you want to avoid a zero mean and unit variance that the StandardScaler results in. 

- The RobustScaler is less influenced by a small number of very large outliers like the other scalers.

It can be hard to pick a scaler though! You'll see later that you can also try them out to see which one gives you the best performance.

</details>

### Further Preprocessing 

Other preprocessing tools from `sklearn.preprocessing` could also be applied to our dataset. For example,
* [**Binarization**](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.Binarizer.html#sklearn.preprocessing.Binarizer): A common operation on text count data where the analyst can decide to only consider the presence or absence of a feature rather than a quantified number of occurrences for instance.
* [**Imputing missing values**](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html#sklearn.impute.SimpleImputer): Address missing values in a column by imputting the mean, median, mode or another constant value.
* [**Generating Polynomial Features**](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html): A technique where you create new features to increase model performance. This transfomer generates a new feature matrix consisting of all polynomial combinations of the features. For example, if an input sample is two dimensional and has features $[a, b]$, the degree-2 polynomial features are $[1, a, b, a^2, ab, b^2]$. This can help the model to learn more complex relationships.
    

All of these are also **Transformers**!. 

For example, 

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(2)

poly.fit_transform(X_train[['fare', 'age']])

### Summary
In this notebook, you saw how to use different kinds of preprocessing steps. In particular, we covered:
- Why **encoding** categorical values is necessary;
- The difference between **ordinal** encoding and **one hot** encoding;
- How to use **Transformers** in Scikit-Learn;
- How to use the **ColumnTransformer** to only apply certain preprocessing steps to **specific** columns;
- Why **scaling** your data is important.